In [37]:
import os
import glob
import pandas as pd
import numpy as np

ptdfs_path = r"C:\Users\UI450907\Desktop\TE RWEST\Tesis\Phase3Results\PTDFs"
limits_path = r"C:\Users\UI450907\Desktop\TE RWEST\Tesis\Phase3Results\Limits"



ptdf_dfs = {}
limits_dfs = {}



In [38]:
def load_csv_folder(folder_path, sep=";", decimal="."):
    """
    Load all CSV files from a folder into a dictionary of DataFrames.

    Parameters
    ----------
    folder_path : str
        Path to the folder containing CSV files.
    sep : str, optional
        CSV separator, by default ";"
    decimal : str, optional
        Decimal separator, by default "."

    Returns
    -------
    dict
        Dictionary where:
        - key = file name without extension
        - value = pandas DataFrame
    """
    csv_files = glob.glob(os.path.join(folder_path, "*.csv"))
    dfs = {}

    for file in csv_files:
        name = os.path.splitext(os.path.basename(file))[0]
        df = pd.read_csv(file, sep=sep, decimal=decimal)
        dfs[name] = df

    return dfs

def filter_ptdf_columns(df):
    """
    This function receives a ptdf result dataframe from powerfactory and clean the columns
    It returns a cleaned dataframe with only the ptdf for lines and trafos of the 
    original dataframe 
    """
    df.rename(columns={df.columns[0]:"Injection Bus"}, inplace=True)

    first_col = df.columns[0] # Keep the bus index
    cols = [first_col]
    seen_elements = set()
    df = df.iloc[1:].reset_index(drop = True)
    for c in df.columns[1:]:
        # keep only lines and trafos
        if not (c.startswith("Line") or c.startswith("Trf")):
            continue

        # remove .1, .2 etc
        if any(f".{i}" in c for i in range(1, 10)):
            continue

        # avoid duplicates (just in case)
        if c not in seen_elements:
            cols.append(c)
            seen_elements.add(c)

    return df[cols]

def reshape_ptdf(df):
    """
    Transform PTDF dataframe from wide to long format.

    Parameters
    ----------
    df : pandas.DataFrame
        PTDF dataframe (wide format)

    Returns
    -------
    pandas.DataFrame
        Long format with columns:
        - Injection Bus
        - Element
        - PTDF
    """
    df_long = df.melt(
        id_vars="Injection Bus",
        var_name="Element",
        value_name="PTDF"
    )

    df_long["PTDF"] = pd.to_numeric(df_long["PTDF"],errors = "coerce")
    df_long["PTDF"] = df_long["PTDF"].fillna(0)

    return df_long

def combine_limits(limits_dfs):
    """
    Combine line and transformer limits into a single dataframe.

    Parameters
    ----------
    limits_dfs : dict
        Dictionary containing limits dataframes
        Expected keys: 'limits_lines', 'limits_trafos'

    Returns
    -------
    pandas.DataFrame
        Combined limits dataframe with unified 'Element' column
    """
    # Copy to avoid modifying original data
    limit_lines = limits_dfs["limits_lines"].copy()
    limit_trafos = limits_dfs["limits_trafos"].copy()

    # Standardize column name
    limit_lines = limit_lines.rename(columns={"name": "Element"})
    limit_trafos = limit_trafos.rename(columns={"name": "Element"})

    # Combine
    limit_elements = pd.concat([limit_lines, limit_trafos], ignore_index=True)

    return limit_elements

def merge_ptdf_with_limits(df_long, limits_df):
    """
    Merge one PTDF long dataframe with the combined limits dataframe.

    Parameters
    ----------
    df_long : pandas.DataFrame
        Long PTDF dataframe with columns like:
        - Injection Bus
        - Element
        - PTDF
    limits_df : pandas.DataFrame
        Combined limits dataframe with column:
        - Element

    Returns
    -------
    pandas.DataFrame
        Merged and sorted dataframe
    """
    result = df_long.merge(limits_df, on="Element", how="left")

    result["Injection Bus"] = pd.to_numeric(
        result["Injection Bus"], errors="coerce"
    )
    result["Injection Bus"] = result["Injection Bus"].astype("Int64")

    result = result.sort_values(by=["Injection Bus", "Element"])
    result = result.reset_index(drop=True)

    return result

def merge_ptdf_with_limits_dict(ptdf_dict,loading_dict):
    results = {}
    for case in ptdf_dict.keys():
        df_ptdf = ptdf_dict[case]
        df_loading = loading_dict[case]

        df = df_ptdf.merge(df_loading,on = "Element", how = "left")
        df["Injection Bus"] = pd.to_numeric(df["Injection Bus"], errors="coerce").astype("Int64")

        df = df.sort_values(by=["Injection Bus", "Element"]).reset_index(drop=True)

        results[case] = df

    return results






def calculate_transfer_limit(df):
    """
    Calculate transfer limit based on PTDF and element limits.

    Parameters
    ----------
    df : pandas.DataFrame
        DataFrame containing:
        - PTDF
        - smax
        - p_used

    Returns
    -------
    pandas.DataFrame
        DataFrame with added 'Transfer Limit' column
    """
    # Calculate transfer limit
    margin = df["smax"] - df["p_used"]
    df["Transfer Limit"] = np.where(
        df["PTDF"] == 0,
        np.inf,  # element not limiting
        (margin/ df["PTDF"]).abs())

    return df


In [39]:
ptdf_dfs = load_csv_folder(ptdfs_path)
limits_dfs = load_csv_folder(limits_path)

ptdf_dfs_filtered = {}
for name,df in ptdf_dfs.items():
    filtered_df = filter_ptdf_columns(df)
    filtered_df = filtered_df.reset_index(drop = True)
    ptdf_dfs_filtered[name] = filtered_df

ptdf_long_dfs = {}
for name, df in ptdf_dfs_filtered.items():
    df = reshape_ptdf(df)
    ptdf_long_dfs[name] = df


limits_combined = combine_limits(limits_dfs)

results = {}
for name,df_long in ptdf_long_dfs.items():
    result = merge_ptdf_with_limits(df_long,limits_combined)
    results[name] = result

for name,result in results.items():
    df = calculate_transfer_limit(result)
    results[name] = df

for name,result in results.items():
    groups = result.groupby("Injection Bus")
    print(f"\n CASE: {name}")
    for bus,df_bus in groups:
        top5 = df_bus.sort_values("Transfer Limit").head(10)
        print(f"\nBus {bus}")
        print(top5[["Element", "PTDF", "Transfer Limit"]])




 CASE: Distribution_Factors_Results_(SYM)_Base

Bus 1
         Element      PTDF  Transfer Limit
35   Trf 06 - 31  1.005349      142.177731
8   Line 05 - 06  0.537879      264.227273
2   Line 02 - 03  0.446103      520.982676
10  Line 06 - 07 -0.257771      651.812421
0   Line 01 - 02  0.549658      858.324198
1   Line 01 - 39  0.450342      976.890920
13  Line 08 - 09 -0.448873     1090.120640
11  Line 06 - 11 -0.208733     1126.074831
4   Line 03 - 04  0.376103     1168.763316
14  Line 09 - 39 -0.449094     1173.092938

Bus 2
         Element      PTDF  Transfer Limit
81   Trf 06 - 31  0.998067      143.215074
54  Line 05 - 06  0.558934      254.273853
48  Line 02 - 03  0.633141      367.077688
50  Line 03 - 04  0.534940      821.728398
57  Line 06 - 11 -0.284849      825.170450
61  Line 10 - 11  0.261502      881.262759
52  Line 04 - 05  0.497678      925.988540
65  Line 15 - 16 -0.248558     1019.358163
56  Line 06 - 07 -0.153282     1096.138748
63  Line 13 - 14 -0.286378     1133

In [40]:
ptdf_dfs_filtered["Distribution_Factors_Results_(SYM)_Base"].head()
#Seller: bus 1 : Hago inyeccion
#Buyer: Bus2: Extraigo
#PTDF_B1 ; PTDF_B2 : PTDF_1->2 = PTDF_1 - PTDF_2 = 0.5496 - (-0.2127) = 
#PTDF 1->2 : Line 01-02, Line 01-39: 

,Injection Bus,Line 01 - 02,Line 01 - 39,Line 02 - 03,Line 02 - 25,Line 03 - 04,Line 03 - 18,Line 04 - 05,Line 04 - 14,Line 05 - 06,...,Trf 10 - 32,Trf 11 - 12,Trf 13 - 12,Trf 19 - 20,Trf 19 - 33,Trf 20 - 34,Trf 22 - 35,Trf 23 - 36,Trf 25 - 37,Trf 29 - 38
0,1.000000,0.549658,0.450342,0.446103,0.107961,0.376103,0.066178,0.343352,0.032026,0.537879,...,----,-0.018670,0.018674,-0.000007,-0.000007,-0.000006,----,-0.000003,----,0.000001
1,2.000000,-0.212760,0.212760,0.633141,0.152401,0.534940,0.092754,0.497678,0.036196,0.558934,...,----,-0.025446,0.025451,-0.000003,-0.000004,-0.000003,----,-0.000001,----,0.000002
2,3.000000,-0.142717,0.142717,-0.171439,0.027586,0.619295,0.210863,0.555742,0.062224,0.576505,...,----,-0.026955,0.026960,-0.000018,-0.000019,-0.000017,----,-0.000007,----,-0.000003
3,4.000000,-0.054118,0.054118,-0.052742,-0.001804,-0.079485,0.027304,0.673412,0.247416,0.638912,...,----,-0.024463,0.024467,-0.000021,-0.000022,-0.000019,----,-0.000008,----,-0.000003
4,5.000000,-0.001083,0.001083,-0.002537,0.001448,-0.007529,0.005079,-0.046312,0.038889,0.864970,...,----,-0.004167,0.004167,-0.000013,-0.000014,-0.000012,----,-0.000005,----,-0.000002


# Logic of the code

1. Read the CSV results from Powerfactory into a dictionary of dataframes : raw_dfs_ptdfs
2. Read the CSV limit results from Powerfactory : raw_dfs_limts
3. Filter the columns of the raw Dataframe of PTDFS to extract only the columns of interest : dfs_ptdfs_filtered
4. Combine the limits into one single Dataframe of limits: dfs_limits_combined
5. Reshape the dfs_ptdfs_filtered dictionary, melt function, transform from wide to long format, id: injection bus, variable: Element, value: PTDF (create function)
6. Rename the dfs_limits_combined dataframe column: "name" to "Element" so merging is possible
7. Merge on element, how: left -> as a result the dictionary of results is created 
8. Transform the relevant columns into the correct type of variable so operating is possible, do it  for each df in the dict of dfs
9. Calculate the transfer limit of each element, for each bus, for each case
10. Order the results to observe the max to the less impacted elements for each bus/case
11. Present the table results

In [41]:
import re
import pandas as pd

def normalize(key):
    key = key.replace("Distribution_Factors_Results_(SYM)_","")
    key = key.replace("Base Case", "Base")
    key = re.sub(r"_+", "_", key)
    return key


def calculate_transfer_limit(df, eps=1e-6):
    df = df.copy()

    margin = df["smax"] - df["actual"]

    df["Transfer Limit"] = np.where(
        margin.notna() & df["PTDF"].notna() & (df["PTDF"] > eps),
        margin / df["PTDF"],
        np.inf
    )

    return df

def calculate_transfer_limit_2(df):
    df = df.copy()

    margin = df["smax"] - df["actual"]

    df["Transfer Limit"] = np.where(
        df["PTDF"] == 0,
        np.inf,  # element not limiting
        (margin / df["PTDF"]).abs()
    )

    return df
def calculate_transfer_limit_3(df):
    df = df.copy()
    df["Limit Used"] = np.sign(df["PTDF"]) * df["smax"]

    margin = df["Limit Used"] - df["actual"]

    df["Transfer Limit"] = np.where(
        df["PTDF"] == 0,
        np.inf,  # element not limiting
        (margin / df["PTDF"]).abs()
    )

    return df
# PTDFS
ptdf_dfs = load_csv_folder(ptdfs_path)
ptdf_dfs_filtered = {}
for name, df in ptdf_dfs.items():
    filtered_df = filter_ptdf_columns(df)
    filtered_df = filtered_df.reset_index(drop=True)
    ptdf_dfs_filtered[name] = filtered_df

ptdf_long_dfs = {}
for name, df in ptdf_dfs_filtered.items():
    df_long = reshape_ptdf(df)
    ptdf_long_dfs[name] = df_long

# LOADINGS FROM CSV
df_loadings = pd.read_csv(r"C:\Users\UI450907\Desktop\TE RWEST\Tesis\Phase3Results\loadings_all.csv")
loading_dict = {}

for case,df_case in df_loadings.groupby("case"):
    df_case = df_case.copy()
    df_case = df_case.rename(columns={"name":"Element"})
    loading_dict[case] = df_case

# build normalized mapping

ptdf_map = {normalize(k): k for k in ptdf_long_dfs.keys()}
loading_map = {normalize(k): k for k in loading_dict.keys()}

valid_cases = set(ptdf_map.keys()) & set(loading_map.keys())

#Merge by case
results = {}

for case in valid_cases:
    ptdf_name = ptdf_map[case]
    loading_name = loading_map[case]

    df_long = ptdf_long_dfs[ptdf_name]
    df_loading = loading_dict[loading_name]

    result = merge_ptdf_with_limits(df_long, df_loading)
    results[case] = result

for name, result in results.items():
    results[name] = calculate_transfer_limit_3(result)


In [42]:
# Top 10 elements per contingncy
for name, result in results.items():
    groups = result.groupby("Injection Bus")
    print(f"\nCASE: {name}")
    for bus, df_bus in groups:
        top10 = df_bus.sort_values("Transfer Limit").head(10)
        print(f"\nBus {bus}")
        print(top10[["Element", "PTDF","Limit Used","actual","Transfer Limit"]])



CASE: Line_21_22_Cnt

Bus 1
         Element      PTDF  Limit Used      actual  Transfer Limit
2   Line 02 - 03  0.444865  597.557529  364.999062      522.761886
1   Line 01 - 39  0.452495  597.557529  122.728259     1049.358048
11  Line 06 - 11 -0.207583 -597.557529 -356.924904     1159.211614
35   Trf 06 - 31  1.005776  700.000000 -530.962257     1223.893050
15  Line 10 - 11  0.190530  597.557529  358.834144     1252.943810
14  Line 09 - 39 -0.451283 -597.557529  -18.550911     1283.023330
13  Line 08 - 09 -0.451123 -597.557529  -18.316462     1283.998083
0   Line 01 - 02  0.547505  597.557529 -122.728259     1315.578465
4   Line 03 - 04  0.376620  597.557529   85.971831     1358.360410
19  Line 15 - 16 -0.173962 -597.557529 -304.231715     1686.148775

Bus 2
         Element      PTDF  Limit Used      actual  Transfer Limit
48  Line 02 - 03  0.631628  597.557529  364.999062      368.188976
57  Line 06 - 11 -0.283308 -597.557529 -356.924904      849.367559
61  Line 10 - 11  0.260103

In [43]:
# All contingencies
all_dfs = []

for case_name,df in results.items():
    df_case = df.copy()
    all_dfs.append(df_case)

df_all = pd.concat(all_dfs, ignore_index=True)

df_all_sorted = df_all.sort_values(["Injection Bus","Transfer Limit"],ascending=[True,True])

top10_per_bus = df_all_sorted.groupby("Injection Bus",group_keys=False).head(10)




# Export the results into csvs

In [44]:
path = r"C:\Users\UI450907\Desktop\TE RWEST\Tesis\Phase3Results\Results"

with pd.ExcelWriter(path +r"\results_bus_by_bus_complete.xlsx", engine = "openpyxl") as writer:
    for bus,df_bus in df_all_sorted.groupby("Injection Bus"):
        sheet_name = f"Bus_{bus}"
        df_bus.to_excel(writer,sheet_name=sheet_name,index = False)

with pd.ExcelWriter(path +r"\results_bus_by_bus_top10.xlsx", engine = "openpyxl") as writer:
    for bus,df_bus in top10_per_bus.groupby("Injection Bus"):
        sheet_name = f"Bus_{bus}"
        df_bus.to_excel(writer,sheet_name=sheet_name,index = False)

In [45]:
grupo = df_all_sorted.groupby("Injection Bus")
grupo.get_group(1).head()

,Injection Bus,Element,PTDF,smax,loading,actual,margin,case,Limit Used,Transfer Limit
1796,1,Line 02 - 03,0.539423,597.557529,96.742803,585.512355,12.045173,Line_26_27_Cnt,597.557529,22.329736
64469,1,Line 02 - 03,0.806360,597.557529,76.838138,464.478099,133.079429,Line_01_39_Cnt,597.557529,165.037241
55543,1,Line 04 - 14,-0.140285,597.557529,95.304579,-566.892616,1164.450144,Line_06_11_Cnt,-597.557529,218.590105
32322,1,Line 26 - 27,0.390010,597.557529,81.988746,497.521086,100.036442,Line_02_03_Cnt,597.557529,256.497122
30500,1,Line 02 - 03,0.811006,597.557529,64.709424,380.955234,216.602295,Line_08_09_Cnt,597.557529,267.078535


# Compare PowerWorld vs Powerfactory

In [46]:
import re
import pandas as pd

def get_injection_bus(s):
    s = str(s)
    match = re.search(r'Bus\s+(\d+)', s)
    if match:
        return int(match.group(1))
    return None


def normalize_element(s):
    s = str(s).strip()
    s_low = s.lower()

    # detectar tipo
    if "transformer" in s_low or "trf" in s_low:
        elem_type = "Trf"
    elif "line" in s_low:
        elem_type = "Line"
    else:
        return None

    # caso PowerWorld: buscar "Bus xx ... Bus yy"
    bus_matches = re.findall(r'Bus\s+(\d+)', s, flags=re.IGNORECASE)

    if len(bus_matches) >= 2:
        b1, b2 = int(bus_matches[0]), int(bus_matches[1])
    else:
        # fallback para formato tipo "Line 02 - 03" o "Trf 12 - 13"
        nums = re.findall(r'\d+', s)
        if len(nums) < 2:
            return None
        b1, b2 = int(nums[0]), int(nums[1])

    b1, b2 = sorted([b1, b2])

    return f"{elem_type} {b1:02d}-{b2:02d}"


def normalize_ctg(s):
    s = str(s).strip()
    s_low = s.lower()

    # Base case
    if "base case" in s_low or s_low == "base_case":
        return "Base Case"

    # detectar tipo
    if s_low.startswith("t_") or "trf" in s_low or "transformer" in s_low:
        elem_type = "Trf"
    elif s_low.startswith("l_") or "line" in s_low:
        elem_type = "Line"
    else:
        return None

    # prioridad 1: buscar buses explícitos
    bus_matches = re.findall(r'Bus\s*(\d+)', s, flags=re.IGNORECASE)

    if len(bus_matches) >= 2:
        b1, b2 = int(bus_matches[0]), int(bus_matches[1])
    else:
        # fallback para formatos tipo Line_26_27_Cnt o Trf_11_12_Cnt
        nums = re.findall(r'\d+', s)
        nums = [int(x) for x in nums]

        if len(nums) < 2:
            return None

        # tomar los últimos dos "números de bus" más probables
        b1, b2 = nums[0], nums[1]

    b1, b2 = sorted([b1, b2])

    return f"{elem_type} {b1:02d}-{b2:02d}_Cnt"

In [47]:
powerworld_path = r"C:\Users\UI450907\Desktop\TE RWEST\Tesis\Phase3Results\PowerWorld\Book1.xlsx"

powerworld = pd.read_excel(powerworld_path, sheet_name= None)

for name,df in powerworld.items():
    df.columns = df.iloc[0]
    df = df.drop(0)
    df = df.reset_index(drop = True)
    df["Injection Bus"] = get_injection_bus(name)
    powerworld[name] = df

#Concatenate in one single dataframe
powerworld_df = pd.concat(powerworld.values(), ignore_index=True)
print(powerworld_df["Injection Bus"].unique())
print(powerworld_df.shape)
powerworld_df["Trans Lim"] = pd.to_numeric(powerworld_df["Trans Lim"], errors="coerce")
powerworld_df["element_key"] = powerworld_df["Limiting Element"].apply(normalize_element)
powerworld_df["ctg_key"] = powerworld_df["Limiting CTG"].apply(normalize_ctg)
powerworld_df.head()

[1 2 7]
(106, 9)


,Trans Lim,Limiting Element,Limiting CTG,% OTDF,% PTDF,Init Value,Pre-Trans Est,Limit Used,Injection Bus,element_key,ctg_key
0,3.66,Line Bus 02 (2) TO Bus 03 (3) CKT 1 [345.00 ...,L_000026Bus26-000027Bus27C1,53.31,43.68,364.26,595.61,597.56,1,Line 02-03,Line 26-27_Cnt
1,167.11,Line Bus 02 (2) TO Bus 03 (3) CKT 1 [345.00 ...,L_000001Bus01-000039Bus39C1,80.02,43.68,364.26,463.84,597.56,1,Line 02-03,Line 01-39_Cnt
2,168.86,Line Bus 04 (4) TO Bus 14 (14) CKT 1 [345.00...,L_000006Bus06-000011Bus11C1,-13.95,3.12,-270.41,-574.00,-597.56,1,Line 04-14,Line 06-11_Cnt
3,254.25,Line Bus 26 (26) TO Bus 27 (27) CKT 1 [345.0...,L_000002Bus02-000003Bus03C1,39.14,10.94,262.95,498.06,597.56,1,Line 26-27,Line 02-03_Cnt
4,271.39,Line Bus 02 (2) TO Bus 03 (3) CKT 1 [345.00 ...,L_000009Bus09-000039Bus39C1,80.02,43.68,364.26,380.39,597.56,1,Line 02-03,Line 09-39_Cnt


In [48]:
df_all_sorted["element_key"] = df_all_sorted["Element"].apply(normalize_element)
df_all_sorted["ctg_key"] = df_all_sorted["case"].apply(normalize_ctg)
df_all_sorted.head(5)


,Injection Bus,Element,PTDF,smax,loading,actual,margin,case,Limit Used,Transfer Limit,element_key,ctg_key
1796,1,Line 02 - 03,0.539423,597.557529,96.742803,585.512355,12.045173,Line_26_27_Cnt,597.557529,22.329736,Line 02-03,Line 26-27_Cnt
64469,1,Line 02 - 03,0.806360,597.557529,76.838138,464.478099,133.079429,Line_01_39_Cnt,597.557529,165.037241,Line 02-03,Line 01-39_Cnt
55543,1,Line 04 - 14,-0.140285,597.557529,95.304579,-566.892616,1164.450144,Line_06_11_Cnt,-597.557529,218.590105,Line 04-14,Line 06-11_Cnt
32322,1,Line 26 - 27,0.390010,597.557529,81.988746,497.521086,100.036442,Line_02_03_Cnt,597.557529,256.497122,Line 26-27,Line 02-03_Cnt
30500,1,Line 02 - 03,0.811006,597.557529,64.709424,380.955234,216.602295,Line_08_09_Cnt,597.557529,267.078535,Line 02-03,Line 08-09_Cnt


In [49]:
dup_pw = powerworld_df.duplicated(
    subset=["Injection Bus", "element_key", "ctg_key"],
    keep=False
)

powerworld_df[dup_pw].sort_values(
    ["Injection Bus", "element_key", "ctg_key"]
).head(20)
print(f"Duplicated in PowerWorld: {dup_pw.sum()}")

dup_pf = df_all_sorted.duplicated(
    subset=["Injection Bus", "element_key", "ctg_key"],
    keep=False
)


print(f"Duplicated in PF: {dup_pf.sum()}")

df_all_sorted[dup_pf].sort_values(
    ["Injection Bus", "element_key", "ctg_key"]
).head(20)
df_pf_clean = df_all_sorted.copy()
df_pf_clean = df_pf_clean.sort_values("Transfer Limit").drop_duplicates(
    subset=["Injection Bus", "element_key", "ctg_key"],
    keep="first"
)
dup_pf_clean = df_pf_clean.duplicated(
    subset=["Injection Bus", "element_key", "ctg_key"],
    keep=False
)

print("Duplicados después:", dup_pf_clean.sum())

Duplicated in PowerWorld: 0
Duplicated in PF: 0
Duplicados después: 0


In [50]:
# Create Rankings
powerworld_df = powerworld_df.sort_values(
    ["Injection Bus", "Trans Lim"],
    ascending=[True, True]
).copy()

powerworld_df["rank_pw"] = powerworld_df.groupby("Injection Bus").cumcount() + 1
df_pf_clean = df_pf_clean.sort_values(
    ["Injection Bus", "Transfer Limit"],
    ascending=[True, True]
).copy()

df_pf_clean["rank_pf"] = df_pf_clean.groupby("Injection Bus").cumcount() + 1


In [51]:
df_pf_clean.head()

,Injection Bus,Element,PTDF,smax,loading,actual,margin,case,Limit Used,Transfer Limit,element_key,ctg_key,rank_pf
1796,1,Line 02 - 03,0.539423,597.557529,96.742803,585.512355,12.045173,Line_26_27_Cnt,597.557529,22.329736,Line 02-03,Line 26-27_Cnt,1
64469,1,Line 02 - 03,0.806360,597.557529,76.838138,464.478099,133.079429,Line_01_39_Cnt,597.557529,165.037241,Line 02-03,Line 01-39_Cnt,2
55543,1,Line 04 - 14,-0.140285,597.557529,95.304579,-566.892616,1164.450144,Line_06_11_Cnt,-597.557529,218.590105,Line 04-14,Line 06-11_Cnt,3
32322,1,Line 26 - 27,0.390010,597.557529,81.988746,497.521086,100.036442,Line_02_03_Cnt,597.557529,256.497122,Line 26-27,Line 02-03_Cnt,4
30500,1,Line 02 - 03,0.811006,597.557529,64.709424,380.955234,216.602295,Line_08_09_Cnt,597.557529,267.078535,Line 02-03,Line 08-09_Cnt,5


In [52]:
# Hacer inner merge para solo  quedarme con casos comparables
df_compare = powerworld_df.merge(
    df_pf_clean,
    on=["Injection Bus", "element_key", "ctg_key"],
    how="inner",
    suffixes=("_pw", "_pf")
)
print(len(powerworld_df))
print(df_compare.shape)
df_compare.head()

106
(98, 22)


,Trans Lim,Limiting Element,Limiting CTG,% OTDF,% PTDF,Init Value,Pre-Trans Est,Limit Used_pw,Injection Bus,element_key,...,Element,PTDF,smax,loading,actual,margin,case,Limit Used_pf,Transfer Limit,rank_pf
0,3.66,Line Bus 02 (2) TO Bus 03 (3) CKT 1 [345.00 ...,L_000026Bus26-000027Bus27C1,53.31,43.68,364.26,595.61,597.56,1,Line 02-03,...,Line 02 - 03,0.539423,597.557529,96.742803,585.512355,12.045173,Line_26_27_Cnt,597.557529,22.329736,1
1,167.11,Line Bus 02 (2) TO Bus 03 (3) CKT 1 [345.00 ...,L_000001Bus01-000039Bus39C1,80.02,43.68,364.26,463.84,597.56,1,Line 02-03,...,Line 02 - 03,0.806360,597.557529,76.838138,464.478099,133.079429,Line_01_39_Cnt,597.557529,165.037241,2
2,168.86,Line Bus 04 (4) TO Bus 14 (14) CKT 1 [345.00...,L_000006Bus06-000011Bus11C1,-13.95,3.12,-270.41,-574.00,-597.56,1,Line 04-14,...,Line 04 - 14,-0.140285,597.557529,95.304579,-566.892616,1164.450144,Line_06_11_Cnt,-597.557529,218.590105,3
3,254.25,Line Bus 26 (26) TO Bus 27 (27) CKT 1 [345.0...,L_000002Bus02-000003Bus03C1,39.14,10.94,262.95,498.06,597.56,1,Line 26-27,...,Line 26 - 27,0.390010,597.557529,81.988746,497.521086,100.036442,Line_02_03_Cnt,597.557529,256.497122,4
4,271.39,Line Bus 02 (2) TO Bus 03 (3) CKT 1 [345.00 ...,L_000009Bus09-000039Bus39C1,80.02,43.68,364.26,380.39,597.56,1,Line 02-03,...,Line 02 - 03,0.811187,597.557529,64.323481,380.880410,216.677118,Line_09_39_Cnt,597.557529,267.111182,6


In [53]:
# Crear vista reducida
df_compare_view = df_compare[
    [
        "Injection Bus",
        "element_key",
        "ctg_key",
        "Trans Lim",
        "Transfer Limit",
        "rank_pw",
        "rank_pf",
    ]
].copy()

# Calcular error porcentual
df_compare_view["error_%"] = (
    (df_compare_view["Transfer Limit"] - df_compare_view["Trans Lim"]).abs()
    / df_compare_view["Trans Lim"].abs()
) * 100

# Diferencia de ranking
df_compare_view["rank_diff"] = (
    df_compare_view["rank_pf"] - df_compare_view["rank_pw"]
)

# Ordenar para visualizar
df_compare_view = df_compare_view.sort_values(
    ["Injection Bus", "rank_pw"],
    ascending=[True, True]
)

# Mostrar resultado
df_compare_view.head(20)

,Injection Bus,element_key,ctg_key,Trans Lim,Transfer Limit,rank_pw,rank_pf,error_%,rank_diff
0,1,Line 02-03,Line 26-27_Cnt,3.66,22.329736,1,1,510.102073,0
1,1,Line 02-03,Line 01-39_Cnt,167.11,165.037241,2,2,1.240356,0
2,1,Line 04-14,Line 06-11_Cnt,168.86,218.590105,3,3,29.450495,0
3,1,Line 26-27,Line 02-03_Cnt,254.25,256.497122,4,4,0.883824,0
4,1,Line 02-03,Line 09-39_Cnt,271.39,267.111182,5,6,1.576631,1
5,1,Line 06-11,Line 04-14_Cnt,298.87,309.921106,6,7,3.697630,1
6,1,Line 10-11,Line 04-14_Cnt,420.72,407.248639,7,10,3.201978,3
7,1,Line 03-04,Line 15-16_Cnt,459.64,468.136529,8,12,1.848518,4
8,1,Line 15-16,Line 03-04_Cnt,491.27,488.696854,9,15,0.523774,6
9,1,Line 06-11,Line 10-13_Cnt,524.70,540.008104,10,35,2.917497,25
